# Parish-level building accessibility workflow

This notebook is a cleaned publication copy of the original parish workflow used in the study.

## How it was used

The same notebook was run separately for each of the seven parishes. For every run, only the
parish name and output prefix were changed. Each parish was divided into four subareas for
computational processing and topology validation. The four final subarea results were then merged
into one final parish result.

## Important

- This notebook preserves the original analytical sequence.
- It does not automate the seven parishes in a loop.
- It does not replace the four-subarea procedure with a different method.
- Intermediate northwest, northeast, southwest and southeast CSV files are outputs of this workflow,
  not source data.
- Change only `PARISH_NAME` and `OUTPUT_PREFIX` before running the notebook for another parish.

In [ ]:
from pathlib import Path

# ---------------------------------------------------------------------
# Run configuration: change these two values for each parish
# ---------------------------------------------------------------------

PARISH_NAME = "Aldoar, Foz do Douro e Nevogilde"
OUTPUT_PREFIX = "aldoar_foz_douro_nevogilde"

PROJECT_ROOT = Path.cwd().resolve().parent
RAW_DIR = PROJECT_ROOT / "data" / "raw"
OUTPUT_DIR = PROJECT_ROOT / "data" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BUILDINGS_FILE = RAW_DIR / "buildings.csv"
SERVICES_FILE = RAW_DIR / "services.csv"
PARISHES_FILE = RAW_DIR / "parishes.csv"
NETWORK_FILE = RAW_DIR / "pedestrian_network_final.csv"
DTM_FILE = RAW_DIR / "dtm_porto_2m.tif"

In [ ]:
# 1. Load data and select the parish

In [ ]:
import pandas as pd
import geopandas as gpd
from shapely import wkt
from shapely.geometry import shape
import json
import folium

def load_csv_with_geometry(path, possible_geom_cols=('geometry_wkt', 'geometry')):
    df = pd.read_csv(path, dtype=str, low_memory=False)

    geom_col = next((c for c in possible_geom_cols if c in df.columns), None)

    if geom_col is not None:
        df['geometry'] = df[geom_col].apply(
            lambda x: wkt.loads(x) if pd.notna(x) and str(x).strip() != '' else None
        )
        gdf = gpd.GeoDataFrame(df, geometry='geometry', crs='EPSG:4326')
        gdf = gdf.dropna(subset=['geometry'])
        return gdf

    return df

def safe_geojson_loads(geojson_str):
    try:
        return shape(json.loads(geojson_str))
    except (ValueError, TypeError, json.JSONDecodeError):
        return None

# File paths
path_edificios = BUILDINGS_FILE
path_servicos = SERVICES_FILE
path_freguesias = PARISHES_FILE
path_rede = NETWORK_FILE

# Load data
edificios = load_csv_with_geometry(path_edificios)
servicos = pd.read_csv(path_servicos, dtype=str, low_memory=False)
freguesias = pd.read_csv(path_freguesias, sep=';', encoding='latin1')
rede = load_csv_with_geometry(path_rede)

if not isinstance(rede, gpd.GeoDataFrame):
    raise ValueError("The file rede_final_ligada.csv does not contain a recognized geometry column.")

# Convert parishes to geometry
freguesias['geometry'] = freguesias['Geo Shape'].apply(safe_geojson_loads)
gdf_freguesias = gpd.GeoDataFrame(
    freguesias,
    geometry='geometry',
    crs='EPSG:4326'
).dropna(subset=['geometry'])

# Freguesia alvo
freguesias_selecionadas = gdf_freguesias[
    gdf_freguesias['Official Name Parish'].str.contains(PARISH_NAME, case=False, na=False)
].copy()

if freguesias_selecionadas.empty:
    raise ValueError(f"{PARISH_NAME} not found.")

print("Selected parish found:")
print(freguesias_selecionadas['Official Name Parish'].drop_duplicates())

# Merge selected parish geometry
freguesias_geom = freguesias_selecionadas.geometry.union_all()

if not isinstance(edificios, gpd.GeoDataFrame):
    raise ValueError("The file edificios.csv does not contain a recognized geometry column.")

# Filter buildings strictly inside the selected parish
edificios_selecionadas = edificios[edificios.geometry.within(freguesias_geom)].copy()

# Convert services to points
servicos['Longitude'] = pd.to_numeric(servicos['Longitude'], errors='coerce')
servicos['Latitude'] = pd.to_numeric(servicos['Latitude'], errors='coerce')
servicos = servicos.dropna(subset=['Longitude', 'Latitude']).copy()

servicos_gdf = gpd.GeoDataFrame(
    servicos,
    geometry=gpd.points_from_xy(servicos['Longitude'], servicos['Latitude']),
    crs='EPSG:4326'
)

# Reproject all layers to metric CRS
freg_proj = freguesias_selecionadas.to_crs(3763)
serv_proj = servicos_gdf.to_crs(3763)
edif_proj = edificios_selecionadas.to_crs(3763)
rede_proj = rede.to_crs(3763)

# 800 m buffer
freguesias_geom_proj = freg_proj.geometry.union_all()
buffer_800m = freguesias_geom_proj.buffer(800)

# Filter services and network
servicos_buffer_800m = serv_proj[serv_proj.geometry.within(buffer_800m)].copy()
rede_selecionada = rede_proj[rede_proj.geometry.intersects(buffer_800m)].copy()

print(f"Buildings in {PARISH_NAME}: {len(edif_proj)}")
print(f"Services within 800 m buffer: {len(servicos_buffer_800m)}")
print(f"Network segments in buffered area: {len(rede_selecionada)}")

# Distance to network
rede_union = rede_selecionada.geometry.union_all()

edif_proj['dist_to_network'] = edif_proj['geometry'].apply(
    lambda geom: geom.centroid.distance(rede_union) if geom.geom_type in ['Polygon', 'MultiPolygon'] else geom.distance(rede_union)
)

edif_proj['ligado_rede'] = edif_proj['dist_to_network'] <= 60

print("Total buildings:", len(edif_proj))
print("Connected buildings:", int(edif_proj['ligado_rede'].sum()))
print("Not connected:", int((~edif_proj['ligado_rede']).sum()))

In [ ]:
# 2. Initial diagnostic map

In [ ]:
edificios_mapa = edif_proj.to_crs(4326)
rede_mapa = rede_selecionada.to_crs(4326)
freguesia_mapa = freguesias_selecionadas.to_crs(4326)

edificios_nao_ligados_mapa = edificios_mapa[~edificios_mapa['ligado_rede']].copy()

centro = [freguesias_geom.centroid.y, freguesias_geom.centroid.x]
m = folium.Map(location=centro, zoom_start=15, tiles="CartoDB positron")

folium.GeoJson(
    freguesia_mapa.geometry.union_all().__geo_interface__,
    name="Aldoar, Foz do Douro e Nevogilde",
    style_function=lambda feature: {
        'fillColor': 'none',
        'color': 'black',
        'weight': 2
    }
).add_to(m)

rede_layer = folium.FeatureGroup(name="Rede pedonal").add_to(m)

for _, row in rede_mapa.iterrows():
    geom = row.geometry

    if geom is None:
        continue

    if geom.geom_type == 'LineString':
        folium.PolyLine(
            locations=[(y, x) for x, y in geom.coords],
            color="blue",
            weight=2,
            opacity=0.7
        ).add_to(rede_layer)

    elif geom.geom_type == 'MultiLineString':
        for linha in geom.geoms:
            folium.PolyLine(
                locations=[(y, x) for x, y in linha.coords],
                color="blue",
                weight=2,
                opacity=0.7
            ).add_to(rede_layer)

nao_ligados_layer = folium.FeatureGroup(name="Edifícios não ligados").add_to(m)

for _, row in edificios_nao_ligados_mapa.iterrows():
    folium.GeoJson(
        data=row.geometry.__geo_interface__,
        style_function=lambda feature: {
            'fillColor': 'red',
            'color': 'red',
            'weight': 1,
            'fillOpacity': 0.7
        },
        tooltip=f"Não ligado | Distância à rede: {row['dist_to_network']:.2f} m"
    ).add_to(nao_ligados_layer)

servicos_layer = folium.FeatureGroup(name="Serviços (buffer 800 m)", show=False).add_to(m)
servicos_mapa = servicos_buffer_800m.to_crs(4326)

for _, row in servicos_mapa.iterrows():
    popup_name = row['Name'] if 'Name' in row.index else 'Serviço'
    folium.Marker(
        location=[row.geometry.y, row.geometry.x],
        popup=popup_name,
        icon=folium.Icon(color='green')
    ).add_to(servicos_layer)

folium.LayerControl().add_to(m)

m

In [ ]:
# 3. Build the graph

In [ ]:
import networkx as nx
from shapely.geometry import LineString

G_total = nx.Graph()

for _, row in rede_selecionada.iterrows():
    geom = row.geometry

    if geom is None:
        continue

    def adicionar_segmentos(linha):
        coords = list(linha.coords)
        for i in range(len(coords) - 1):
            u = coords[i]
            v = coords[i + 1]
            segmento = LineString([u, v])
            comprimento = segmento.length
            G_total.add_edge(u, v, length=comprimento)

    if geom.geom_type == "LineString":
        adicionar_segmentos(geom)

    elif geom.geom_type == "MultiLineString":
        for linha in geom.geoms:
            adicionar_segmentos(linha)

print("Graph created.")
print("Nodes:", G_total.number_of_nodes())
print("Edges:", G_total.number_of_edges())

In [ ]:
# 4. Load the DTM and assign elevation

In [ ]:
import rasterio
import numpy as np

mdt_path = DTM_FILE
mdt_src = rasterio.open(mdt_path)

print("MDT loaded.")
print("CRS:", mdt_src.crs)

def obter_altitude_no(x, y, raster_src):
    try:
        row, col = raster_src.index(x, y)
        valor = raster_src.read(1)[row, col]

        if raster_src.nodata is not None and valor == raster_src.nodata:
            return None

        if np.isnan(valor):
            return None

        return float(valor)
    except Exception:
        return None

altitudes_nos = {}

for node in G_total.nodes:
    x, y = node
    altitudes_nos[node] = obter_altitude_no(x, y, mdt_src)

print("Altitudes assigned to graph nodes.")

In [ ]:
# 5. Create the slope-adjusted travel-time attribute

In [ ]:
velocidade_base = 0.7  # m/s

def calcular_travel_time_slope(node_u, node_v, edge_data, altitudes_nos, velocidade_base=0.7):
    length = edge_data.get('length', None)

    if length is None or length <= 0:
        return None

    z1 = altitudes_nos.get(node_u, None)
    z2 = altitudes_nos.get(node_v, None)

    if z1 is None or z2 is None:
        return length / velocidade_base

    delta_z = z2 - z1
    slope = delta_z / length
    tempo_plano = length / velocidade_base

    if slope > 0:
        fator_subida = 1 + slope
    else:
        fator_subida = 1

    return tempo_plano * fator_subida

for u, v, data in G_total.edges(data=True):
    travel_time = calcular_travel_time_slope(
        u, v, data, altitudes_nos, velocidade_base=velocidade_base
    )

    if travel_time is None:
        length = data.get('length', 0)
        travel_time = length / velocidade_base if length > 0 else None

    data['travel_time_slope'] = travel_time

print("Attribute 'travel_time_slope' created.")

In [ ]:
# 6. Divide the parish into four processing subareas

In [ ]:
from shapely.geometry import box

minx, miny, maxx, maxy = freguesias_geom_proj.bounds
mid_x = (minx + maxx) / 2
mid_y = (maxy + miny) / 2

noroeste_geom = freguesias_geom_proj.intersection(box(minx, mid_y, mid_x, maxy))
nordeste_geom = freguesias_geom_proj.intersection(box(mid_x, mid_y, maxx, maxy))
sudoeste_geom = freguesias_geom_proj.intersection(box(minx, miny, mid_x, mid_y))
sudeste_geom = freguesias_geom_proj.intersection(box(mid_x, miny, maxx, mid_y))

In [ ]:
# 7. Base function for processing one subarea

In [ ]:
from scipy.spatial import cKDTree
from joblib import Parallel, delayed
import pandas as pd
import networkx as nx

categorias_alvo = [
    'Centro Saude',
    'Farmacias',
    'Hospitais',
    'Supermercados',
    'Bancos',
    'CTT',
    'Parques e jardins'
]

col_categoria = 'Category'

# ============================================================
# USAR APENAS A COMPONENTE PRINCIPAL DO GRAFO PARA O SNAP
# E PARA O CÁLCULO DOS CAMINHOS
# ============================================================

componentes = list(nx.connected_components(G_total))
componentes_ordenadas = sorted(componentes, key=len, reverse=True)
componente_principal = componentes_ordenadas[0]

print(f"Número total de componentes: {len(componentes)}")
print(f"Tamanho da componente principal: {len(componente_principal)}")

G_main = G_total.subgraph(componente_principal).copy()

nos_lista = list(G_main.nodes)
arvore_nos = cKDTree(nos_lista)

def encontrar_no_mais_proximo_grafo(geom, arvore_nos, nos_lista):
    if geom.geom_type == 'Point':
        ponto_ref = geom
    elif geom.geom_type in ['Polygon', 'MultiPolygon']:
        ponto_ref = geom.centroid
    else:
        return None

    _, idx = arvore_nos.query((ponto_ref.x, ponto_ref.y))
    return nos_lista[idx]

def calcular_metricas_edificio(nodo_edificio, servicos_gdf, distancia_max=800, tempo_max_seg=1143):
    distancias_validas = []
    tempos_validos = []
    todas_distancias = []
    contagens = {cat: 0 for cat in categorias_alvo}

    distancia_minima_servico = None
    tempo_minimo_servico_seg = None
    servico_min_id = None
    servico_min_categoria = None

    for _, servico in servicos_gdf.iterrows():
        nodo_servico = servico['nearest_node']
        categoria = servico[col_categoria]

        if 'Name' in servico.index and pd.notna(servico['Name']):
            servico_id_atual = servico['Name']
        elif 'osm_id' in servico.index and pd.notna(servico['osm_id']):
            servico_id_atual = servico['osm_id']
        else:
            servico_id_atual = None

        try:
            distancia = nx.shortest_path_length(
                G_main,
                nodo_edificio,
                nodo_servico,
                weight='length'
            )
            tempo = nx.shortest_path_length(
                G_main,
                nodo_edificio,
                nodo_servico,
                weight='travel_time_slope'
            )

            todas_distancias.append(distancia)

            # ====================================================
            # CORREÇÃO MÍNIMA DE ZEROS ARTIFICIAIS
            # ====================================================
            if (distancia_minima_servico is None) or (distancia < distancia_minima_servico):
                distancia_ajustada = distancia
                tempo_ajustado = tempo

                # evitar colapso no mesmo nó (distância/tempo = 0 artificiais)
                if distancia == 0 and tempo == 0:
                    distancia_ajustada = 5
                    tempo_ajustado = 5

                distancia_minima_servico = distancia_ajustada
                tempo_minimo_servico_seg = tempo_ajustado
                servico_min_id = servico_id_atual
                servico_min_categoria = categoria

            if (distancia <= distancia_max) and (tempo <= tempo_max_seg):
                distancias_validas.append(distancia)
                tempos_validos.append(tempo)

                if categoria in contagens:
                    contagens[categoria] += 1

        except (nx.NetworkXNoPath, nx.NodeNotFound):
            continue

    numero_servicos_proximos = sum(contagens.values())
    servicos_por_categoria = {k: v for k, v in contagens.items() if v > 0}

    if distancias_validas:
        distancia_media_servicos = sum(distancias_validas) / len(distancias_validas)
        tempo_medio_servicos_seg = sum(tempos_validos) / len(tempos_validos)
    else:
        distancia_media_servicos = distancia_minima_servico
        tempo_medio_servicos_seg = None

    return {
        'distancia_media_servicos': distancia_media_servicos,
        'distancia_minima_servico': distancia_minima_servico,
        'tempo_medio_servicos_seg': tempo_medio_servicos_seg,
        'tempo_minimo_servico_seg': tempo_minimo_servico_seg,
        'servico_min_id': servico_min_id,
        'servico_min_categoria': servico_min_categoria,
        'servicos_por_categoria': servicos_por_categoria,
        'numero_servicos_proximos': numero_servicos_proximos,
        'Centro Saude': contagens['Centro Saude'],
        'Farmacias': contagens['Farmacias'],
        'Hospitais': contagens['Hospitais'],
        'Supermercados': contagens['Supermercados'],
        'Bancos': contagens['Bancos'],
        'CTT': contagens['CTT'],
        'Parques e jardins': contagens['Parques e jardins']
    }

def processar_subarea(geom_subarea, nome_saida):
    edificios_validos = edif_proj[edif_proj['ligado_rede']].copy()
    edificios_sub = edificios_validos[edificios_validos.geometry.within(geom_subarea)].copy()

    servicos_validos = servicos_buffer_800m[
        servicos_buffer_800m[col_categoria].isin(categorias_alvo)
    ].copy()

    edificios_sub['nearest_node'] = edificios_sub['geometry'].apply(
        lambda geom: encontrar_no_mais_proximo_grafo(geom, arvore_nos, nos_lista)
    )

    servicos_validos['nearest_node'] = servicos_validos['geometry'].apply(
        lambda geom: encontrar_no_mais_proximo_grafo(geom, arvore_nos, nos_lista)
    )

    edificios_sub['component_id'] = 0
    servicos_validos['component_id'] = 0

    componentes_com_servicos = {0}

    print(f"{nome_saida} - before filtering: {len(edificios_sub)}")

    edificios_sub = edificios_sub[
        edificios_sub['component_id'].isin(componentes_com_servicos)
    ].copy()

    print(f"{nome_saida} - after filtering: {len(edificios_sub)}")

    resultados = Parallel(n_jobs=-1)(
        delayed(calcular_metricas_edificio)(
            nodo_edificio,
            servicos_validos,
            distancia_max=800,
            tempo_max_seg=1143
        )
        for nodo_edificio in edificios_sub['nearest_node']
    )

    resultados_df = pd.DataFrame(resultados, index=edificios_sub.index)
    edificios_resultado = edificios_sub.join(resultados_df)

    print(f"{nome_saida} completed. Buildings: {len(edificios_resultado)}")
    return edificios_resultado

# 8A. Correção topológica **inline por subárea**

Esta secção é executada **logo após cada subárea**, e **antes** de passar para a subárea seguinte.

Fluxo correto por subárea:

**1. cálculo base da subárea → 2. CSV base da subárea → 3. deteção de suspeitos → 4. hotspots → 5. teste de ligações candidatas → 6. aplicação das ligações aceites → 7. recálculo da subárea → 8. CSV final da subárea**

No fim das 4 subáreas:
- `#Unir`
- validação
- mapa final

In [ ]:

import ast
import itertools
import math
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import networkx as nx

from shapely import wkt
from shapely.geometry import Point, LineString

GRAPH_CRS = "EPSG:3763"

# -----------------------------
# parâmetros metodológicos
# -----------------------------
SUSPECT_RATIO_THRESHOLD = 1.5
SUSPECT_EXCESS_THRESHOLD = 50.0

CORRIDOR_BUFFER_M = 35
MAX_LINK_DIST_M = 20
TOP_K_CANDIDATES = 20

MIN_CASES_IMPROVED = 5
MIN_MEAN_IMPROVEMENT = 30.0
MAX_ITERS_PER_HOTSPOT = 5

HOTSPOT_GRID_SIZE_M = 150
MIN_CASES_PER_HOTSPOT = 5

SERVICES_CSV = str(SERVICES_FILE)



# =========================================================
# UTILITÁRIOS
# =========================================================
def safe_wkt_load(v):
    if pd.isna(v):
        return None
    s = str(v).strip()
    if not s:
        return None
    try:
        return wkt.loads(s)
    except Exception:
        return None

def safe_tuple_to_point(v):
    if pd.isna(v):
        return None
    s = str(v).strip()
    if not s:
        return None
    try:
        t = ast.literal_eval(s)
        if isinstance(t, (tuple, list)) and len(t) >= 2:
            x, y = float(t[0]), float(t[1])
            if np.isfinite(x) and np.isfinite(y):
                return Point(x, y)
    except Exception:
        return None
    return None

def valid_geom(g):
    if g is None or not hasattr(g, "is_empty") or g.is_empty:
        return False
    try:
        return all(np.isfinite(v) for v in g.bounds)
    except Exception:
        return False

def geom_to_point(g):
    if not valid_geom(g):
        return None
    if g.geom_type == "Point":
        return g
    try:
        rp = g.representative_point()
        if valid_geom(rp):
            return rp
    except Exception:
        pass
    return None

def get_node_xy(node, data=None):
    if data is None:
        data = {}

    if isinstance(data, dict):
        if "x" in data and "y" in data:
            return float(data["x"]), float(data["y"])
        if "lon" in data and "lat" in data:
            return float(data["lon"]), float(data["lat"])
        if "X" in data and "Y" in data:
            return float(data["X"]), float(data["Y"])
        if "geometry" in data and data["geometry"] is not None:
            geom = data["geometry"]
            if hasattr(geom, "geom_type") and geom.geom_type == "Point" and not geom.is_empty:
                return float(geom.x), float(geom.y)

    if isinstance(node, (tuple, list)) and len(node) >= 2:
        try:
            return float(node[0]), float(node[1])
        except Exception:
            pass

    return None

def nearest_graph_node_to_point(G, point_xy):
    px, py = point_xy
    best_node = None
    best_d2 = None

    for n, data in G.nodes(data=True):
        xy = get_node_xy(n, data)
        if xy is None:
            continue

        nx_, ny_ = xy
        d2 = (nx_ - px) ** 2 + (ny_ - py) ** 2

        if best_d2 is None or d2 < best_d2:
            best_d2 = d2
            best_node = n

    return best_node

def shortest_path_nodes(G, source, target):
    try:
        return nx.shortest_path(G, source=source, target=target, weight="length")
    except Exception:
        return nx.shortest_path(G, source=source, target=target)

def path_to_linestring(G, path_nodes):
    coords = []
    for n in path_nodes:
        xy = get_node_xy(n, G.nodes[n])
        if xy is None:
            continue
        coords.append(xy)

    if len(coords) < 2:
        return None

    try:
        return LineString(coords)
    except Exception:
        return None

def route_length_between_nodes(G, source, target):
    try:
        path_nodes = shortest_path_nodes(G, source, target)
        path_line = path_to_linestring(G, path_nodes)
        if path_line is None or path_line.is_empty:
            return float("nan"), None, None
        return float(path_line.length), path_nodes, path_line
    except Exception:
        return float("nan"), None, None



# =========================================================
# DIAGNÓSTICO DA ÁREA
# =========================================================
def load_services_gdf(services_csv=SERVICES_CSV):
    serv = pd.read_csv(services_csv)
    serv = serv.dropna(subset=["Longitude", "Latitude"]).copy()
    serv["Name_norm"] = serv["Name"].astype(str).str.strip().str.lower()

    return gpd.GeoDataFrame(
        serv,
        geometry=gpd.points_from_xy(serv["Longitude"], serv["Latitude"]),
        crs="EPSG:4326"
    ).to_crs(GRAPH_CRS)

def load_area_results_for_diagnostics(area_csv, services_gdf):
    df = pd.read_csv(area_csv).copy()
    df["_nearest_node_pt"] = df["nearest_node"].apply(safe_tuple_to_point)
    df["_geom_edificio"] = df["geometry_wkt"].apply(safe_wkt_load)

    def choose_origin(row):
        if valid_geom(row["_nearest_node_pt"]):
            return row["_nearest_node_pt"]
        return geom_to_point(row["_geom_edificio"])

    df["_origem_atual"] = df.apply(choose_origin, axis=1)
    df["servico_min_id_norm"] = df["servico_min_id"].astype(str).str.strip().str.lower()

    serv_join = services_gdf[["Name_norm", "Category", "geometry"]].copy()
    serv_join = serv_join.rename(columns={"geometry": "_destino_geom", "Category": "_destino_categoria_real"})

    merged = df.merge(
        serv_join,
        left_on="servico_min_id_norm",
        right_on="Name_norm",
        how="left"
    )

    # nomes repetidos: escolher match mais próximo da origem
    if len(merged) > len(df):
        merged = gpd.GeoDataFrame(merged, geometry="_destino_geom", crs=GRAPH_CRS)
        merged["_dist_match"] = merged.apply(
            lambda r: r["_origem_atual"].distance(r["_destino_geom"])
            if valid_geom(r["_origem_atual"]) and valid_geom(r["_destino_geom"])
            else np.nan,
            axis=1
        )
        merged = merged.sort_values(["osm_id", "_dist_match"])
        merged = merged.groupby("osm_id", as_index=False).first()

    merged = gpd.GeoDataFrame(merged, geometry="_destino_geom", crs=GRAPH_CRS)

    merged["_dist_direta_m"] = merged.apply(
        lambda r: r["_origem_atual"].distance(r["_destino_geom"])
        if valid_geom(r["_origem_atual"]) and valid_geom(r["_destino_geom"])
        else np.nan,
        axis=1
    )
    merged["_dist_rede_m"] = pd.to_numeric(merged["distancia_minima_servico"], errors="coerce")
    merged["_excesso_abs_m"] = merged["_dist_rede_m"] - merged["_dist_direta_m"]
    merged["_ratio_rede_direta"] = merged["_dist_rede_m"] / merged["_dist_direta_m"]

    merged["_flag_suspeito"] = (
        merged["_dist_direta_m"].notna() &
        (merged["_dist_direta_m"] > 0) &
        (merged["_ratio_rede_direta"] > SUSPECT_RATIO_THRESHOLD) &
        (merged["_excesso_abs_m"] > SUSPECT_EXCESS_THRESHOLD)
    )

    return merged

def hotspot_grid_labels(cases_df, cell_size_m=HOTSPOT_GRID_SIZE_M):
    tmp = cases_df.copy()
    tmp["_x"] = tmp["_origem_atual"].apply(lambda g: g.x if valid_geom(g) else np.nan)
    tmp["_y"] = tmp["_origem_atual"].apply(lambda g: g.y if valid_geom(g) else np.nan)
    tmp["_gx"] = np.floor(tmp["_x"] / cell_size_m).astype("Int64")
    tmp["_gy"] = np.floor(tmp["_y"] / cell_size_m).astype("Int64")
    tmp["hotspot_id"] = tmp["_gx"].astype(str) + "_" + tmp["_gy"].astype(str)
    return tmp

def detect_hotspots(area_diag_df, min_cases_per_hotspot=MIN_CASES_PER_HOTSPOT):
    suspects = area_diag_df[area_diag_df["_flag_suspeito"]].copy()
    suspects = suspects[suspects["_origem_atual"].notna()].copy()
    if len(suspects) == 0:
        return suspects, pd.DataFrame(columns=["hotspot_id", "n_cases"])

    suspects = hotspot_grid_labels(suspects, HOTSPOT_GRID_SIZE_M)

    counts = (
        suspects.groupby("hotspot_id", as_index=False)
        .size()
        .rename(columns={"size": "n_cases"})
        .sort_values("n_cases", ascending=False)
    )

    keep = counts[counts["n_cases"] >= min_cases_per_hotspot]["hotspot_id"].tolist()
    suspects = suspects[suspects["hotspot_id"].isin(keep)].copy()
    counts = counts[counts["hotspot_id"].isin(keep)].copy()

    return suspects, counts



# =========================================================
# GERAÇÃO E TESTE DE CANDIDATAS POR HOTSPOT
# =========================================================
def point_in_polygon(pt, polygon_geom):
    if not valid_geom(pt):
        return False
    return pt.within(polygon_geom) or pt.intersects(polygon_geom)

def build_case_direct_line(row):
    origem = row["_origem_atual"]
    destino = row["_destino_geom"]
    if valid_geom(origem) and valid_geom(destino):
        try:
            return LineString([origem, destino])
        except Exception:
            return None
    return None

def build_corridor_from_cases(cases_df, buffer_m=CORRIDOR_BUFFER_M):
    geoms = []
    for _, row in cases_df.iterrows():
        line = build_case_direct_line(row)
        if line is not None and not line.is_empty:
            geoms.append(line.buffer(buffer_m))
    if not geoms:
        return None
    return gpd.GeoSeries(geoms, crs=GRAPH_CRS).union_all()

def extract_subgraph_nodes_in_geom(G, polygon_geom):
    inside_nodes = []
    for n, data in G.nodes(data=True):
        xy = get_node_xy(n, data)
        if xy is None:
            continue
        p = Point(xy)
        if p.within(polygon_geom) or p.intersects(polygon_geom):
            inside_nodes.append(n)
    return inside_nodes

def make_nodes_gdf(G_sub):
    rows = []
    for n, data in G_sub.nodes(data=True):
        xy = get_node_xy(n, data)
        if xy is None:
            continue
        rows.append({
            "node": n,
            "degree": G_sub.degree(n),
            "geometry": Point(xy)
        })
    return gpd.GeoDataFrame(rows, geometry="geometry", crs=GRAPH_CRS)

def find_endpoint_candidates_in_corridor(G, cases_df):
    corridor = build_corridor_from_cases(cases_df, CORRIDOR_BUFFER_M)
    if corridor is None or corridor.is_empty:
        return gpd.GeoDataFrame(columns=["node_a", "node_b", "dist_m", "score", "geometry"], geometry="geometry", crs=GRAPH_CRS)

    sub_nodes = extract_subgraph_nodes_in_geom(G, corridor)
    G_sub = G.subgraph(sub_nodes).copy()

    nodes_gdf = make_nodes_gdf(G_sub)
    endpoints = nodes_gdf[nodes_gdf["degree"] == 1].copy()

    rows = []
    recs = endpoints.to_dict("records")

    for a, b in itertools.combinations(recs, 2):
        pa = a["geometry"]
        pb = b["geometry"]
        d = pa.distance(pb)

        if d == 0 or d > MAX_LINK_DIST_M:
            continue

        na = a["node"]
        nb = b["node"]

        if G_sub.has_edge(na, nb) or G_sub.has_edge(nb, na):
            continue

        seg = LineString([pa, pb])
        score = seg.distance(corridor) + d

        rows.append({
            "node_a": na,
            "node_b": nb,
            "dist_m": d,
            "score": score,
            "geometry": seg
        })

    cand = gpd.GeoDataFrame(rows, geometry="geometry", crs=GRAPH_CRS)
    if len(cand) == 0:
        return cand

    return cand.sort_values(["score", "dist_m"], ascending=[True, True]).reset_index(drop=True)

def evaluate_candidate_on_cases(G_base, cases_df, node_a, node_b, link_dist_m):
    G_tmp = G_base.copy()
    G_tmp.add_edge(node_a, node_b, length=link_dist_m)
    G_tmp.add_edge(node_b, node_a, length=link_dist_m)

    rows = []

    for _, row in cases_df.iterrows():
        origem = row["_origem_atual"]
        destino = row["_destino_geom"]

        if not valid_geom(origem) or not valid_geom(destino):
            continue

        origem_node_before = nearest_graph_node_to_point(G_base, (origem.x, origem.y))
        destino_node_before = nearest_graph_node_to_point(G_base, (destino.x, destino.y))

        origem_node_after = nearest_graph_node_to_point(G_tmp, (origem.x, origem.y))
        destino_node_after = nearest_graph_node_to_point(G_tmp, (destino.x, destino.y))

        dist_before, _, _ = route_length_between_nodes(G_base, origem_node_before, destino_node_before)
        dist_after, _, _ = route_length_between_nodes(G_tmp, origem_node_after, destino_node_after)

        melhoria = (
            dist_before - dist_after
            if pd.notna(dist_before) and pd.notna(dist_after)
            else np.nan
        )

        rows.append({
            "osm_id": row["osm_id"],
            "dist_before": dist_before,
            "dist_after": dist_after,
            "melhoria_m": melhoria
        })

    eval_df = pd.DataFrame(rows)

    n_improved = int((eval_df["melhoria_m"] > 0).sum())
    mean_improvement = float(eval_df.loc[eval_df["melhoria_m"] > 0, "melhoria_m"].mean()) if n_improved > 0 else 0.0
    median_improvement = float(eval_df.loc[eval_df["melhoria_m"] > 0, "melhoria_m"].median()) if n_improved > 0 else 0.0
    total_improvement = float(eval_df["melhoria_m"].clip(lower=0).sum()) if len(eval_df) else 0.0

    return {
        "node_a": node_a,
        "node_b": node_b,
        "link_dist_m": link_dist_m,
        "n_cases": len(eval_df),
        "n_improved": n_improved,
        "mean_improvement_m": mean_improvement,
        "median_improvement_m": median_improvement,
        "total_improvement_m": total_improvement,
        "eval_df": eval_df,
        "G_tmp": G_tmp
    }

def search_best_candidate_for_hotspot(G_base, cases_df):
    cand = find_endpoint_candidates_in_corridor(G_base, cases_df)
    if len(cand) == 0:
        return None, cand, pd.DataFrame()

    tested = []
    for _, c in cand.head(TOP_K_CANDIDATES).iterrows():
        result = evaluate_candidate_on_cases(
            G_base,
            cases_df,
            c["node_a"],
            c["node_b"],
            c["dist_m"]
        )

        tested.append({
            "node_a": result["node_a"],
            "node_b": result["node_b"],
            "link_dist_m": result["link_dist_m"],
            "n_cases": result["n_cases"],
            "n_improved": result["n_improved"],
            "mean_improvement_m": result["mean_improvement_m"],
            "median_improvement_m": result["median_improvement_m"],
            "total_improvement_m": result["total_improvement_m"],
        })

    tested_df = pd.DataFrame(tested)
    tested_df = tested_df.sort_values(
        ["n_improved", "total_improvement_m", "mean_improvement_m"],
        ascending=[False, False, False]
    ).reset_index(drop=True)

    best = tested_df.iloc[0].to_dict() if len(tested_df) else None
    return best, cand, tested_df

def iterative_hotspot_fix(G_start, cases_df):
    G_current = G_start.copy()
    accepted_links = []
    remaining_cases = cases_df.copy()

    for it in range(1, MAX_ITERS_PER_HOTSPOT + 1):
        best, cand_df, tested_df = search_best_candidate_for_hotspot(G_current, remaining_cases)

        print(f"\n=== Iteração {it} ===")
        print("Casos em avaliação:", len(remaining_cases))

        if best is None:
            print("Sem candidatas úteis.")
            break

        display(tested_df.head(10))

        if (
            best["n_improved"] < MIN_CASES_IMPROVED or
            best["mean_improvement_m"] < MIN_MEAN_IMPROVEMENT
        ):
            print("A melhor candidata já não compensa. Parar.")
            break

        print("Aceite:")
        print(best)

        G_current.add_edge(best["node_a"], best["node_b"], length=best["link_dist_m"])
        G_current.add_edge(best["node_b"], best["node_a"], length=best["link_dist_m"])

        accepted_links.append({"iter": it, **best})

        result = evaluate_candidate_on_cases(
            G_start if it == 1 else G_current,
            remaining_cases,
            best["node_a"],
            best["node_b"],
            best["link_dist_m"]
        )

        eval_df = result["eval_df"][["osm_id", "melhoria_m"]].copy()

        remaining_cases = remaining_cases.merge(eval_df, on="osm_id", how="left")
        remaining_cases = remaining_cases[(remaining_cases["melhoria_m"].fillna(0) <= 0)].copy()
        remaining_cases = remaining_cases.drop(columns=["melhoria_m"])

        if len(remaining_cases) == 0:
            print("Todos os casos do hotspot ficaram resolvidos ou melhorados.")
            break

    accepted_df = pd.DataFrame(accepted_links)
    return G_current, accepted_df, remaining_cases


# =========================================================
# ADAPTAÇÃO AO FLUXO ATUAL DO NOTEBOOK
# =========================================================
def set_routing_graph_context(G_graph):
    """Atualiza o contexto global usado por processar_subarea sem alterar a função original."""
    global G_main, nos_lista, arvore_nos
    G_main = G_graph.copy()
    nos_lista = list(G_main.nodes)
    arvore_nos = cKDTree(nos_lista)
    return G_main

def processar_subarea_with_graph(geom_subarea, nome_saida, G_graph):
    """Corre processar_subarea usando temporariamente um grafo fornecido."""
    global G_main, nos_lista, arvore_nos
    old_G_main = G_main
    old_nos_lista = nos_lista
    old_arvore_nos = arvore_nos

    try:
        set_routing_graph_context(G_graph)
        return processar_subarea(geom_subarea, nome_saida)
    finally:
        G_main = old_G_main
        nos_lista = old_nos_lista
        arvore_nos = old_arvore_nos

def save_area_result_csv(gdf_result, csv_path):
    out = gdf_result.copy()
    out["geometry_wkt"] = out.geometry.to_wkt()
    out.drop(columns="geometry").to_csv(csv_path, index=False)



def print_area_correction_report(area_name, area_diag_df, suspects_df, hotspot_counts_df, accepted_links_df, final_result_gdf=None):
    print("\n" + "="*80)
    print(f"RELATÓRIO DA ÁREA: {area_name.upper()}")
    print("="*80)

    total_registos = len(area_diag_df) if area_diag_df is not None else 0
    total_suspeitos = len(suspects_df) if suspects_df is not None else 0
    perc = (100.0 * total_suspeitos / total_registos) if total_registos else 0.0

    print("Resumo base da área")
    print("- Total registos:", total_registos)
    print("- Suspeitos:", total_suspeitos)
    print("- Percentagem suspeitos:", round(perc, 2), "%")

    if area_diag_df is not None and len(area_diag_df) > 0:
        dist_media = pd.to_numeric(area_diag_df.get("_dist_rede_m"), errors="coerce").mean()
        ratio_med = pd.to_numeric(area_diag_df.get("_ratio_rede_direta"), errors="coerce").median()
        print("- Distância média base (m):", round(dist_media, 3) if pd.notna(dist_media) else None)
        print("- Mediana ratio rede/direta:", round(ratio_med, 3) if pd.notna(ratio_med) else None)

    print("\nHotspots")
    if hotspot_counts_df is not None and len(hotspot_counts_df) > 0:
        print("- Nº hotspots:", len(hotspot_counts_df))
        display(hotspot_counts_df.head(10))
    else:
        print("- Nenhum hotspot encontrado.")

    print("\nLigações aceites")
    if accepted_links_df is not None and len(accepted_links_df) > 0:
        print("- Nº ligações aceites:", len(accepted_links_df))
        print("- Casos melhorados (soma n_improved):", int(pd.to_numeric(accepted_links_df.get("n_improved"), errors="coerce").fillna(0).sum()))
        mean_imp = pd.to_numeric(accepted_links_df.get("mean_improvement_m"), errors="coerce").mean()
        total_imp = pd.to_numeric(accepted_links_df.get("total_improvement_m"), errors="coerce").sum()
        print("- Melhoria média por ligação (m):", round(mean_imp, 3) if pd.notna(mean_imp) else None)
        print("- Melhoria total acumulada (m):", round(total_imp, 3) if pd.notna(total_imp) else None)
        display(accepted_links_df)
    else:
        print("- Nenhuma ligação aceite nesta área.")

    if final_result_gdf is not None:
        print("\nResultado final da área")
        print("- Registos finais:", len(final_result_gdf))
        if "distancia_minima_servico" in final_result_gdf.columns:
            d = pd.to_numeric(final_result_gdf["distancia_minima_servico"], errors="coerce")
            print("- Distância média final (m):", round(d.mean(), 3) if pd.notna(d.mean()) else None)
            print("- Distância mediana final (m):", round(d.median(), 3) if pd.notna(d.median()) else None)


def run_inline_topology_correction(area_name, geom_subarea, nome_saida, base_csv, final_csv=None, services_csv=SERVICES_CSV):
    """Aplica correção topológica à subárea e recalcula a própria subárea no mesmo ponto do fluxo."""
    if final_csv is None:
        final_csv = base_csv

    G_area_base = G_main.copy()

    print(f"\n>>> Início da correção topológica inline: {area_name}")
    print("CSV base:", base_csv)
    print("CSV final:", final_csv)

    G_corr, accepted_links_df, area_diag_df, suspects_df, hotspot_counts_df = run_topology_correction_for_area(
        area_name=area_name,
        area_csv=base_csv,
        G_area=G_area_base,
        services_csv=services_csv
    )

    print_area_correction_report(
        area_name=area_name,
        area_diag_df=area_diag_df,
        suspects_df=suspects_df,
        hotspot_counts_df=hotspot_counts_df,
        accepted_links_df=accepted_links_df,
        final_result_gdf=None,
    )

    print(f"\n>>> Recalcular subárea {area_name} com grafo corrigido...")
    resultado_corr = processar_subarea_with_graph(
        geom_subarea=geom_subarea,
        nome_saida=f"{nome_saida}_corr",
        G_graph=G_corr
    )

    print_area_correction_report(
        area_name=f"{area_name} (após recálculo)",
        area_diag_df=area_diag_df,
        suspects_df=suspects_df,
        hotspot_counts_df=hotspot_counts_df,
        accepted_links_df=accepted_links_df,
        final_result_gdf=resultado_corr,
    )

    print(f"\n>>> Guardar CSV final corrigido da área {area_name}...")
    save_area_result_csv(resultado_corr, final_csv)
    print("CSV final gravado em:", final_csv)

    return {
        "resultado_corr": resultado_corr,
        "G_corr": G_corr,
        "accepted_links_df": accepted_links_df,
        "area_diag_df": area_diag_df,
        "suspects_df": suspects_df,
        "hotspot_counts_df": hotspot_counts_df,
        "base_csv": base_csv,
        "final_csv": final_csv,
    }


In [ ]:
# 8. Run one subarea at a time

In [ ]:
#Noroeste
edificios_noroeste_aldoar_foz_douro_nevogilde_resultado = processar_subarea(noroeste_geom, "northwest_aldoar_foz_douro_nevogilde")

save_area_result_csv(
    edificios_noroeste_aldoar_foz_douro_nevogilde_resultado,
    "aldoar_foz_douro_nevogilde_noroeste_base.csv"
)

In [ ]:
# =========================================================
# GERAÇÃO E TESTE DE CANDIDATAS POR HOTSPOT
# =========================================================
def point_in_polygon(pt, polygon_geom):
    if not valid_geom(pt):
        return False
    return pt.within(polygon_geom) or pt.intersects(polygon_geom)

def build_case_direct_line(row):
    origem = row["_origem_atual"]
    destino = row["_destino_geom"]
    if valid_geom(origem) and valid_geom(destino):
        try:
            return LineString([origem, destino])
        except Exception:
            return None
    return None

def build_corridor_from_cases(cases_df, buffer_m=CORRIDOR_BUFFER_M):
    geoms = []
    for _, row in cases_df.iterrows():
        line = build_case_direct_line(row)
        if line is not None and not line.is_empty:
            geoms.append(line.buffer(buffer_m))
    if not geoms:
        return None
    return gpd.GeoSeries(geoms, crs=GRAPH_CRS).union_all()

def extract_subgraph_nodes_in_geom(G, polygon_geom):
    inside_nodes = []
    for n, data in G.nodes(data=True):
        xy = get_node_xy(n, data)
        if xy is None:
            continue
        p = Point(xy)
        if p.within(polygon_geom) or p.intersects(polygon_geom):
            inside_nodes.append(n)
    return inside_nodes

def make_nodes_gdf(G_sub):
    rows = []
    for n, data in G_sub.nodes(data=True):
        xy = get_node_xy(n, data)
        if xy is None:
            continue
        rows.append({
            "node": n,
            "degree": G_sub.degree(n),
            "geometry": Point(xy)
        })

    if len(rows) == 0:
        return gpd.GeoDataFrame(
            columns=["node", "degree", "geometry"],
            geometry="geometry",
            crs=GRAPH_CRS
        )

    return gpd.GeoDataFrame(rows, geometry="geometry", crs=GRAPH_CRS)

def find_endpoint_candidates_in_corridor(G, cases_df):
    corridor = build_corridor_from_cases(cases_df, CORRIDOR_BUFFER_M)
    if corridor is None or corridor.is_empty:
        return gpd.GeoDataFrame(
            columns=["node_a", "node_b", "dist_m", "score", "geometry"],
            geometry="geometry",
            crs=GRAPH_CRS
        )

    sub_nodes = extract_subgraph_nodes_in_geom(G, corridor)
    G_sub = G.subgraph(sub_nodes).copy()

    nodes_gdf = make_nodes_gdf(G_sub)
    if len(nodes_gdf) == 0:
        return gpd.GeoDataFrame(
            columns=["node_a", "node_b", "dist_m", "score", "geometry"],
            geometry="geometry",
            crs=GRAPH_CRS
        )

    endpoints = nodes_gdf[nodes_gdf["degree"] == 1].copy()
    if len(endpoints) < 2:
        return gpd.GeoDataFrame(
            columns=["node_a", "node_b", "dist_m", "score", "geometry"],
            geometry="geometry",
            crs=GRAPH_CRS
        )

    rows = []
    recs = endpoints.to_dict("records")

    for a, b in itertools.combinations(recs, 2):
        pa = a["geometry"]
        pb = b["geometry"]
        d = pa.distance(pb)

        if d == 0 or d > MAX_LINK_DIST_M:
            continue

        na = a["node"]
        nb = b["node"]

        if G_sub.has_edge(na, nb) or G_sub.has_edge(nb, na):
            continue

        seg = LineString([pa, pb])
        score = seg.distance(corridor) + d

        rows.append({
            "node_a": na,
            "node_b": nb,
            "dist_m": d,
            "score": score,
            "geometry": seg
        })

    if len(rows) == 0:
        return gpd.GeoDataFrame(
            columns=["node_a", "node_b", "dist_m", "score", "geometry"],
            geometry="geometry",
            crs=GRAPH_CRS
        )

    cand = gpd.GeoDataFrame(rows, geometry="geometry", crs=GRAPH_CRS)
    return cand.sort_values(["score", "dist_m"], ascending=[True, True]).reset_index(drop=True)

def evaluate_candidate_on_cases(G_base, cases_df, node_a, node_b, link_dist_m):
    G_tmp = G_base.copy()
    G_tmp.add_edge(node_a, node_b, length=link_dist_m)
    G_tmp.add_edge(node_b, node_a, length=link_dist_m)

    rows = []

    for _, row in cases_df.iterrows():
        origem = row["_origem_atual"]
        destino = row["_destino_geom"]

        if not valid_geom(origem) or not valid_geom(destino):
            continue

        origem_node_before = nearest_graph_node_to_point(G_base, (origem.x, origem.y))
        destino_node_before = nearest_graph_node_to_point(G_base, (destino.x, destino.y))

        origem_node_after = nearest_graph_node_to_point(G_tmp, (origem.x, origem.y))
        destino_node_after = nearest_graph_node_to_point(G_tmp, (destino.x, destino.y))

        dist_before, _, _ = route_length_between_nodes(G_base, origem_node_before, destino_node_before)
        dist_after, _, _ = route_length_between_nodes(G_tmp, origem_node_after, destino_node_after)

        melhoria = (
            dist_before - dist_after
            if pd.notna(dist_before) and pd.notna(dist_after)
            else np.nan
        )

        rows.append({
            "osm_id": row["osm_id"],
            "dist_before": dist_before,
            "dist_after": dist_after,
            "melhoria_m": melhoria
        })

    eval_df = pd.DataFrame(rows)

    n_improved = int((eval_df["melhoria_m"] > 0).sum()) if len(eval_df) else 0
    mean_improvement = float(eval_df.loc[eval_df["melhoria_m"] > 0, "melhoria_m"].mean()) if n_improved > 0 else 0.0
    median_improvement = float(eval_df.loc[eval_df["melhoria_m"] > 0, "melhoria_m"].median()) if n_improved > 0 else 0.0
    total_improvement = float(eval_df["melhoria_m"].clip(lower=0).sum()) if len(eval_df) else 0.0

    return {
        "node_a": node_a,
        "node_b": node_b,
        "link_dist_m": link_dist_m,
        "n_cases": len(eval_df),
        "n_improved": n_improved,
        "mean_improvement_m": mean_improvement,
        "median_improvement_m": median_improvement,
        "total_improvement_m": total_improvement,
        "eval_df": eval_df,
        "G_tmp": G_tmp
    }

def search_best_candidate_for_hotspot(G_base, cases_df):
    cand = find_endpoint_candidates_in_corridor(G_base, cases_df)

    if cand is None or len(cand) == 0:
        return None, cand, pd.DataFrame(
            columns=[
                "node_a", "node_b", "link_dist_m", "n_cases", "n_improved",
                "mean_improvement_m", "median_improvement_m", "total_improvement_m"
            ]
        )

    tested = []

    for _, c in cand.head(TOP_K_CANDIDATES).iterrows():
        result = evaluate_candidate_on_cases(
            G_base,
            cases_df,
            c["node_a"],
            c["node_b"],
            c["dist_m"]
        )

        tested.append({
            "node_a": result["node_a"],
            "node_b": result["node_b"],
            "link_dist_m": result["link_dist_m"],
            "n_cases": result["n_cases"],
            "n_improved": result["n_improved"],
            "mean_improvement_m": result["mean_improvement_m"],
            "median_improvement_m": result["median_improvement_m"],
            "total_improvement_m": result["total_improvement_m"],
        })

    if len(tested) == 0:
        return None, cand, pd.DataFrame(
            columns=[
                "node_a", "node_b", "link_dist_m", "n_cases", "n_improved",
                "mean_improvement_m", "median_improvement_m", "total_improvement_m"
            ]
        )

    tested_df = pd.DataFrame(tested)
    tested_df = tested_df.sort_values(
        ["n_improved", "total_improvement_m", "mean_improvement_m"],
        ascending=[False, False, False]
    ).reset_index(drop=True)

    best = tested_df.iloc[0].to_dict()
    return best, cand, tested_df

def iterative_hotspot_fix(G_start, cases_df):
    G_current = G_start.copy()
    accepted_links = []
    remaining_cases = cases_df.copy()

    for it in range(1, MAX_ITERS_PER_HOTSPOT + 1):
        best, cand_df, tested_df = search_best_candidate_for_hotspot(G_current, remaining_cases)

        print(f"\n=== Iteração {it} ===")
        print("Casos em avaliação:", len(remaining_cases))

        if best is None:
            print("Sem candidatas úteis.")
            break

        display(tested_df.head(10))

        if (
            best["n_improved"] < MIN_CASES_IMPROVED or
            best["mean_improvement_m"] < MIN_MEAN_IMPROVEMENT
        ):
            print("A melhor candidata já não compensa. Parar.")
            break

        print("Aceite:")
        print(best)

        G_current.add_edge(best["node_a"], best["node_b"], length=best["link_dist_m"])
        G_current.add_edge(best["node_b"], best["node_a"], length=best["link_dist_m"])

        accepted_links.append({"iter": it, **best})

        result = evaluate_candidate_on_cases(
            G_start if it == 1 else G_current,
            remaining_cases,
            best["node_a"],
            best["node_b"],
            best["link_dist_m"]
        )

        eval_df = result["eval_df"][["osm_id", "melhoria_m"]].copy()

        remaining_cases = remaining_cases.merge(eval_df, on="osm_id", how="left")
        remaining_cases = remaining_cases[(remaining_cases["melhoria_m"].fillna(0) <= 0)].copy()
        remaining_cases = remaining_cases.drop(columns=["melhoria_m"])

        if len(remaining_cases) == 0:
            print("Todos os casos do hotspot ficaram resolvidos ou melhorados.")
            break

    accepted_df = pd.DataFrame(accepted_links)
    return G_current, accepted_df, remaining_cases

In [ ]:
# =========================================================
# VALIDAÇÃO DO SERVIÇO ESCOLHIDO
# =========================================================
def validate_service_selection(
    result_gdf,
    services_csv=SERVICES_CSV,
    graph_crs=GRAPH_CRS,
    ratio_threshold=2.5,
    excess_threshold=150.0,
    max_direct_fallback_m=80.0,
    replace_invalid=False
):
    """
    Valida o serviço final escolhido comparando distância de rede vs distância direta.
    Por defeito apenas sinaliza casos implausíveis; não substitui automaticamente o serviço.
    """
    out = result_gdf.copy()

    def safe_parse_nearest_node(v):
        if pd.isna(v):
            return None
        s = str(v).strip()
        if not s:
            return None
        try:
            t = ast.literal_eval(s)
            if isinstance(t, (tuple, list)) and len(t) >= 2:
                x, y = float(t[0]), float(t[1])
                if np.isfinite(x) and np.isfinite(y):
                    return Point(x, y)
        except Exception:
            pass
        return None

    if "nearest_node" in out.columns:
        out["_origem_pt"] = out["nearest_node"].apply(safe_parse_nearest_node)
    else:
        out["_origem_pt"] = None

    if "_origem_pt" not in out.columns or out["_origem_pt"].isna().all():
        if "geometry" in out.columns:
            out["_origem_pt"] = out.geometry.apply(geom_to_point)
        else:
            out["_origem_pt"] = None

    serv = pd.read_csv(services_csv)
    serv = serv.dropna(subset=["Longitude", "Latitude"]).copy()
    serv["Name_norm"] = serv["Name"].astype(str).str.strip().str.lower()

    serv_gdf = gpd.GeoDataFrame(
        serv,
        geometry=gpd.points_from_xy(serv["Longitude"], serv["Latitude"]),
        crs="EPSG:4326"
    ).to_crs(graph_crs)

    serv_join = serv_gdf[["Name_norm", "Name", "Category", "geometry"]].copy()
    serv_join = serv_join.rename(columns={
        "geometry": "_destino_geom",
        "Name": "_destino_nome_real",
        "Category": "_destino_categoria_real"
    })

    out["servico_min_id_norm"] = out["servico_min_id"].astype(str).str.strip().str.lower()

    merged = out.merge(
        serv_join,
        left_on="servico_min_id_norm",
        right_on="Name_norm",
        how="left"
    )

    if len(merged) > len(out):
        merged = gpd.GeoDataFrame(merged, geometry="geometry", crs=graph_crs)
        merged["_dist_match"] = merged.apply(
            lambda r: r["_origem_pt"].distance(r["_destino_geom"])
            if valid_geom(r["_origem_pt"]) and valid_geom(r["_destino_geom"])
            else np.nan,
            axis=1
        )
        merged = merged.sort_values(["osm_id", "_dist_match"])
        merged = merged.groupby("osm_id", as_index=False).first()

    merged = gpd.GeoDataFrame(merged, geometry="geometry", crs=result_gdf.crs)

    merged["_dist_direta_servico_escolhido_m"] = merged.apply(
        lambda r: r["_origem_pt"].distance(r["_destino_geom"])
        if valid_geom(r["_origem_pt"]) and valid_geom(r["_destino_geom"])
        else np.nan,
        axis=1
    )

    merged["_dist_rede_servico_escolhido_m"] = pd.to_numeric(
        merged["distancia_minima_servico"], errors="coerce"
    )

    merged["_ratio_rede_direta"] = (
        merged["_dist_rede_servico_escolhido_m"] /
        merged["_dist_direta_servico_escolhido_m"]
    )

    merged["_excesso_rede_vs_direta_m"] = (
        merged["_dist_rede_servico_escolhido_m"] -
        merged["_dist_direta_servico_escolhido_m"]
    )

    merged["ligacao_servico_valida"] = ~(
        merged["_dist_direta_servico_escolhido_m"].notna() &
        (merged["_dist_direta_servico_escolhido_m"] > 0) &
        (
            (merged["_ratio_rede_direta"] > ratio_threshold) &
            (merged["_excesso_rede_vs_direta_m"] > excess_threshold)
        )
    )

    def invalid_reason(row):
        if pd.isna(row["_dist_direta_servico_escolhido_m"]):
            return "sem_match_servico"
        if row["ligacao_servico_valida"]:
            return None
        return "ratio_excesso_alto"

    merged["motivo_invalidade_servico"] = merged.apply(invalid_reason, axis=1)

    alt_names = []
    alt_cats = []
    alt_dists = []

    serv_points = serv_gdf[["Name", "Category", "geometry"]].copy()

    for _, row in merged.iterrows():
        origem = row["_origem_pt"]
        if not valid_geom(origem):
            alt_names.append(None)
            alt_cats.append(None)
            alt_dists.append(np.nan)
            continue

        d = serv_points.geometry.distance(origem)
        idx = d.idxmin()

        alt_names.append(serv_points.loc[idx, "Name"])
        alt_cats.append(serv_points.loc[idx, "Category"])
        alt_dists.append(float(d.loc[idx]))

    merged["servico_candidato_direto"] = alt_names
    merged["categoria_candidato_direto"] = alt_cats
    merged["dist_direta_candidato_m"] = alt_dists

    if replace_invalid:
        use_fallback = (
            (~merged["ligacao_servico_valida"]) &
            merged["dist_direta_candidato_m"].notna() &
            (merged["dist_direta_candidato_m"] <= max_direct_fallback_m) &
            (
                merged["_dist_direta_servico_escolhido_m"].isna() |
                (merged["dist_direta_candidato_m"] < merged["_dist_direta_servico_escolhido_m"] * 0.6)
            )
        )

        merged["servico_min_id_original"] = merged["servico_min_id"]
        merged["servico_min_categoria_original"] = merged["servico_min_categoria"]

        merged.loc[use_fallback, "servico_min_id"] = merged.loc[use_fallback, "servico_candidato_direto"]
        merged.loc[use_fallback, "servico_min_categoria"] = merged.loc[use_fallback, "categoria_candidato_direto"]
        merged.loc[use_fallback, "metodo_escolha_servico"] = "fallback_direto"
        merged.loc[~use_fallback, "metodo_escolha_servico"] = "rede"
    else:
        merged["metodo_escolha_servico"] = np.where(
            merged["ligacao_servico_valida"], "rede", "rede_invalida_sinalizada"
        )

    print("\n--- Validação do serviço escolhido ---")
    print("Total registos:", len(merged))
    n_invalid = int((~merged["ligacao_servico_valida"]).sum())
    print("Ligações inválidas:", n_invalid)
    print("Percentagem inválidas:", round(100 * n_invalid / len(merged), 2) if len(merged) else 0, "%")

    if n_invalid > 0:
        cols_show = [
            "osm_id", "servico_min_id", "servico_min_categoria",
            "distancia_minima_servico", "_dist_direta_servico_escolhido_m",
            "_ratio_rede_direta", "_excesso_rede_vs_direta_m",
            "servico_candidato_direto", "categoria_candidato_direto",
            "dist_direta_candidato_m", "motivo_invalidade_servico"
        ]
        display(
            merged.loc[~merged["ligacao_servico_valida"], cols_show]
            .sort_values(["_ratio_rede_direta", "_excesso_rede_vs_direta_m"], ascending=False)
            .head(20)
        )

    return gpd.GeoDataFrame(merged, geometry="geometry", crs=result_gdf.crs)

In [ ]:
def run_inline_topology_correction(area_name, geom_subarea, nome_saida, base_csv, final_csv=None, services_csv=SERVICES_CSV):
    """Aplica correção topológica à subárea e recalcula a própria subárea no mesmo ponto do fluxo."""
    if final_csv is None:
        final_csv = base_csv

    G_area_base = G_main.copy()

    print(f"\n>>> Início da correção topológica inline: {area_name}")
    print("CSV base:", base_csv)
    print("CSV final:", final_csv)

    G_corr, accepted_links_df, area_diag_df, suspects_df, hotspot_counts_df = run_topology_correction_for_area(
        area_name=area_name,
        area_csv=base_csv,
        G_area=G_area_base,
        services_csv=services_csv
    )

    print_area_correction_report(
        area_name=area_name,
        area_diag_df=area_diag_df,
        suspects_df=suspects_df,
        hotspot_counts_df=hotspot_counts_df,
        accepted_links_df=accepted_links_df,
        final_result_gdf=None,
    )

    print(f"\n>>> Recalcular subárea {area_name} com grafo corrigido...")
    resultado_corr = processar_subarea_with_graph(
        geom_subarea=geom_subarea,
        nome_saida=f"{nome_saida}_corr",
        G_graph=G_corr
    )

    print(f"\n>>> Validar serviço escolhido na área {area_name}...")
    resultado_corr = validate_service_selection(
        result_gdf=resultado_corr,
        services_csv=services_csv,
        graph_crs=GRAPH_CRS,
        ratio_threshold=2.5,
        excess_threshold=150.0,
        max_direct_fallback_m=80.0,
        replace_invalid=False
    )

    print_area_correction_report(
        area_name=f"{area_name} (após recálculo)",
        area_diag_df=area_diag_df,
        suspects_df=suspects_df,
        hotspot_counts_df=hotspot_counts_df,
        accepted_links_df=accepted_links_df,
        final_result_gdf=resultado_corr,
    )

    print(f"\n>>> Guardar CSV final corrigido da área {area_name}...")
    save_area_result_csv(resultado_corr, final_csv)
    print("CSV final gravado em:", final_csv)

    return {
        "resultado_corr": resultado_corr,
        "G_corr": G_corr,
        "accepted_links_df": accepted_links_df,
        "area_diag_df": area_diag_df,
        "suspects_df": suspects_df,
        "hotspot_counts_df": hotspot_counts_df,
        "base_csv": base_csv,
        "final_csv": final_csv,
    }

In [ ]:
# =========================================================
# Correção topológica por área
# =========================================================
def run_topology_correction_for_area(area_name, area_csv, G_area, services_csv=SERVICES_CSV):
    services_gdf = load_services_gdf(services_csv)
    area_diag_df = load_area_results_for_diagnostics(area_csv, services_gdf)

    suspects_df, hotspot_counts_df = detect_hotspots(
        area_diag_df,
        min_cases_per_hotspot=MIN_CASES_PER_HOTSPOT
    )

    G_corr = G_area.copy()
    accepted_all = []

    if len(hotspot_counts_df) == 0:
        print(f"[{area_name}] Nenhum hotspot encontrado.")
        accepted_links_df = pd.DataFrame()
        return G_corr, accepted_links_df, area_diag_df, suspects_df, hotspot_counts_df

    print(f"[{area_name}] Hotspots encontrados: {len(hotspot_counts_df)}")

    for _, hrow in hotspot_counts_df.iterrows():
        hotspot_id = hrow["hotspot_id"]
        hotspot_cases = suspects_df[suspects_df["hotspot_id"] == hotspot_id].copy()

        print(f"\n[{area_name}] Hotspot {hotspot_id} | casos: {len(hotspot_cases)}")

        G_corr, accepted_df, remaining_cases = iterative_hotspot_fix(
            G_start=G_corr,
            cases_df=hotspot_cases
        )

        if accepted_df is not None and len(accepted_df) > 0:
            accepted_df = accepted_df.copy()
            accepted_df["area_name"] = area_name
            accepted_df["hotspot_id"] = hotspot_id
            accepted_all.append(accepted_df)

    if len(accepted_all) > 0:
        accepted_links_df = pd.concat(accepted_all, ignore_index=True)
    else:
        accepted_links_df = pd.DataFrame()

    return G_corr, accepted_links_df, area_diag_df, suspects_df, hotspot_counts_df

In [ ]:
corr_noroeste = run_inline_topology_correction(
    area_name="noroeste",
    geom_subarea=noroeste_geom,
    nome_saida="northwest_aldoar_foz_douro_nevogilde",
    base_csv="aldoar_foz_douro_nevogilde_noroeste_base.csv",
    final_csv="aldoar_foz_douro_nevogilde_noroeste.csv",
    services_csv=SERVICES_CSV
)

In [ ]:
# Nordeste

In [ ]:
#Nordeste
edificios_nordeste_aldoar_foz_douro_nevogilde_resultado = processar_subarea(nordeste_geom, "northeast_aldoar_foz_douro_nevogilde")

save_area_result_csv(
    edificios_nordeste_aldoar_foz_douro_nevogilde_resultado,
    "aldoar_foz_douro_nevogilde_nordeste_base.csv"
)

In [ ]:
# Correção topológica inline - nordeste
corr_nordeste = run_inline_topology_correction(
    area_name="nordeste",
    geom_subarea=nordeste_geom,
    nome_saida="northeast_aldoar_foz_douro_nevogilde",
    base_csv="aldoar_foz_douro_nevogilde_nordeste_base.csv",
    final_csv="aldoar_foz_douro_nevogilde_nordeste.csv",
    services_csv=SERVICES_CSV
)

accepted_links_nordeste = corr_nordeste["accepted_links_df"]
area_diag_nordeste = corr_nordeste["area_diag_df"]
suspects_nordeste = corr_nordeste["suspects_df"]
hotspot_counts_nordeste = corr_nordeste["hotspot_counts_df"]
G_nordeste_corr = corr_nordeste["G_corr"]
edificios_nordeste_aldoar_foz_douro_nevogilde_resultado_corr = corr_nordeste["resultado_corr"]

print("nordeste: CSV final corrigido gravado em", corr_nordeste["final_csv"])
print("nordeste: ligações aceites =", len(accepted_links_nordeste))

print("nordeste: hotspots encontrados =", len(hotspot_counts_nordeste))
if len(hotspot_counts_nordeste) > 0:
    display(hotspot_counts_nordeste.head(10))
print("nordeste: suspeitos =", len(suspects_nordeste))
print("nordeste: ligações aceites =", len(accepted_links_nordeste))
if len(accepted_links_nordeste) > 0:
    display(accepted_links_nordeste)


In [ ]:
#Sudoeste

In [ ]:
#Sudoeste
edificios_sudoeste_aldoar_foz_douro_nevogilde_resultado = processar_subarea(sudoeste_geom, "southwest_aldoar_foz_douro_nevogilde")

save_area_result_csv(
    edificios_sudoeste_aldoar_foz_douro_nevogilde_resultado,
    "aldoar_foz_douro_nevogilde_sudoeste_base.csv"
)

In [ ]:
# Correção topológica inline - sudoeste
corr_sudoeste = run_inline_topology_correction(
    area_name="sudoeste",
    geom_subarea=sudoeste_geom,
    nome_saida="southwest_aldoar_foz_douro_nevogilde",
    base_csv="aldoar_foz_douro_nevogilde_sudoeste_base.csv",
    final_csv="aldoar_foz_douro_nevogilde_sudoeste.csv",
    services_csv=SERVICES_CSV
)

accepted_links_sudoeste = corr_sudoeste["accepted_links_df"]
area_diag_sudoeste = corr_sudoeste["area_diag_df"]
suspects_sudoeste = corr_sudoeste["suspects_df"]
hotspot_counts_sudoeste = corr_sudoeste["hotspot_counts_df"]
G_sudoeste_corr = corr_sudoeste["G_corr"]
edificios_sudoeste_aldoar_foz_douro_nevogilde_resultado_corr = corr_sudoeste["resultado_corr"]

print("sudoeste: CSV final corrigido gravado em", corr_sudoeste["final_csv"])
print("sudoeste: ligações aceites =", len(accepted_links_sudoeste))

print("sudoeste: hotspots encontrados =", len(hotspot_counts_sudoeste))
if len(hotspot_counts_sudoeste) > 0:
    display(hotspot_counts_sudoeste.head(10))
print("sudoeste: suspeitos =", len(suspects_sudoeste))
print("sudoeste: ligações aceites =", len(accepted_links_sudoeste))
if len(accepted_links_sudoeste) > 0:
    display(accepted_links_sudoeste)


In [ ]:
#sudeste

In [ ]:
#Sudeste
edificios_sudeste_aldoar_foz_douro_nevogilde_resultado = processar_subarea(sudeste_geom, "southeast_aldoar_foz_douro_nevogilde")

save_area_result_csv(
    edificios_sudeste_aldoar_foz_douro_nevogilde_resultado,
    "aldoar_foz_douro_nevogilde_sudeste_base.csv"
)

In [ ]:
# Correção topológica inline - sudeste
corr_sudeste = run_inline_topology_correction(
    area_name="sudeste",
    geom_subarea=sudeste_geom,
    nome_saida="southeast_aldoar_foz_douro_nevogilde",
    base_csv="aldoar_foz_douro_nevogilde_sudeste_base.csv",
    final_csv="aldoar_foz_douro_nevogilde_sudeste.csv",
    services_csv=SERVICES_CSV
)

accepted_links_sudeste = corr_sudeste["accepted_links_df"]
area_diag_sudeste = corr_sudeste["area_diag_df"]
suspects_sudeste = corr_sudeste["suspects_df"]
hotspot_counts_sudeste = corr_sudeste["hotspot_counts_df"]
G_sudeste_corr = corr_sudeste["G_corr"]
edificios_sudeste_aldoar_foz_douro_nevogilde_resultado_corr = corr_sudeste["resultado_corr"]

print("sudeste: CSV final corrigido gravado em", corr_sudeste["final_csv"])
print("sudeste: ligações aceites =", len(accepted_links_sudeste))

print("sudeste: hotspots encontrados =", len(hotspot_counts_sudeste))
if len(hotspot_counts_sudeste) > 0:
    display(hotspot_counts_sudeste.head(10))
print("sudeste: suspeitos =", len(suspects_sudeste))
print("sudeste: ligações aceites =", len(accepted_links_sudeste))
if len(accepted_links_sudeste) > 0:
    display(accepted_links_sudeste)


In [ ]:
#Unir

In [ ]:
import pandas as pd

df_noroeste = pd.read_csv("aldoar_foz_douro_nevogilde_noroeste.csv")
df_nordeste = pd.read_csv("aldoar_foz_douro_nevogilde_nordeste.csv")
df_sudoeste = pd.read_csv("aldoar_foz_douro_nevogilde_sudoeste.csv")
df_sudeste = pd.read_csv("aldoar_foz_douro_nevogilde_sudeste.csv")

resultados_aldoar_foz_douro_nevogilde_unidos = pd.concat(
    [df_noroeste, df_nordeste, df_sudoeste, df_sudeste],
    axis=0,
    ignore_index=True
)

print("Total de registos unidos:", len(resultados_aldoar_foz_douro_nevogilde_unidos))
print("Duplicados em osm_id:", resultados_aldoar_foz_douro_nevogilde_unidos["osm_id"].duplicated().sum())
print("Tem geometry_wkt?", "geometry_wkt" in resultados_aldoar_foz_douro_nevogilde_unidos.columns)

resultados_aldoar_foz_douro_nevogilde_unidos.to_csv(
    "resultados_distancia_media_servicos_800m_19min_aldoar_foz_douro_nevogilde_global.csv",
    index=False
)

print("Ficheiro final de Aldoar, Foz do Douro e Nevogilde gravado com sucesso.")

In [ ]:
#10. Validar

In [ ]:
df = pd.read_csv("resultados_distancia_media_servicos_800m_19min_aldoar_foz_douro_nevogilde_global.csv")

print("Número de linhas:", len(df))
print("Colunas:", df.columns.tolist())
print("Duplicados em osm_id:", df["osm_id"].duplicated().sum())

casos_inconsistentes = df[
    (df["numero_servicos_proximos"] == 0) &
    (df["tempo_medio_servicos_seg"].notna())
]

print("Casos inconsistentes:", len(casos_inconsistentes))

print(df[[
    "distancia_media_servicos",
    "tempo_medio_servicos_seg",
    "numero_servicos_proximos"
]].describe())

In [ ]:
#Mapa

In [ ]:
import pandas as pd
import geopandas as gpd
from shapely import wkt

df = pd.read_csv("resultados_distancia_media_servicos_800m_19min_aldoar_foz_douro_nevogilde_global.csv")

print("Número de linhas:", len(df))
print("Tem geometry_wkt?", "geometry_wkt" in df.columns)

df["geometry"] = df["geometry_wkt"].apply(
    lambda x: wkt.loads(x) if pd.notna(x) else None
)

resultados_aldoar_foz_douro_nevogilde_geo = gpd.GeoDataFrame(
    df,
    geometry="geometry",
    crs="EPSG:3763"
)

print(resultados_aldoar_foz_douro_nevogilde_geo.head())
print("CRS:", resultados_aldoar_foz_douro_nevogilde_geo.crs)

In [ ]:
import folium
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import geopandas as gpd
import ast
from shapely import wkt
from matplotlib.colors import to_hex
from branca.colormap import LinearColormap

# ============================================================
# INPUTS
# ============================================================
csv_resultados = "resultados_distancia_media_servicos_800m_19min_aldoar_foz_douro_nevogilde_global.csv"
csv_servicos = "servicos.csv"

# variáveis disponíveis para seleção no LayerControl
VARIAVEIS_MAPA = {
    "Distância média aos serviços": "distancia_media_servicos",
    "Distância mínima ao serviço": "distancia_minima_servico",
    "Tempo médio aos serviços": "tempo_medio_servicos_seg",
    "Tempo mínimo ao serviço": "tempo_minimo_servico_seg",
    "Número de serviços próximos": "numero_servicos_proximos",
    "Centro de Saúde": "Centro Saude",
    "Farmácias": "Farmacias",
    "Hospitais": "Hospitais",
    "Supermercados": "Supermercados",
    "Bancos": "Bancos",
    "CTT": "CTT",
    "Parques e jardins": "Parques e jardins",
}

# qual camada fica ligada por defeito
VAR_DEFAULT = "Distância média aos serviços"

# ============================================================
# 1. FUNÇÕES AUXILIARES
# ============================================================
def limpar_valor(v, max_len=120):
    if v is None:
        return "N/A"

    try:
        if pd.isna(v):
            return "N/A"
    except Exception:
        pass

    if isinstance(v, str):
        txt = v.strip()
        if txt == "":
            return "N/A"

        if (txt.startswith("{") and txt.endswith("}")) or (txt.startswith("[") and txt.endswith("]")):
            try:
                v = ast.literal_eval(txt)
            except Exception:
                return txt[:max_len]
        else:
            return txt[:max_len]

    if isinstance(v, dict):
        partes = []
        for k, val in list(v.items())[:10]:
            partes.append(f"{k}: {val}")
        return " | ".join(partes)[:max_len]

    if isinstance(v, (list, tuple)):
        partes = []
        for item in list(v)[:10]:
            if isinstance(item, dict):
                if "value" in item:
                    partes.append(str(item["value"]))
                elif "name" in item:
                    partes.append(str(item["name"]))
                elif "primary" in item:
                    partes.append(str(item["primary"]))
                else:
                    partes.append(str(item))
            else:
                partes.append(str(item))
        return " | ".join(partes)[:max_len]

    return str(v)[:max_len]

def fmt_metros(valor):
    return f"{valor:.2f} m" if pd.notna(valor) else "N/A"

def fmt_segundos(valor):
    return f"{valor:.2f} s" if pd.notna(valor) else "N/A"

def fmt_minutos_from_seg(valor_seg):
    return f"{valor_seg/60:.2f} min" if pd.notna(valor_seg) else "N/A"

def fmt_int(valor, default="0"):
    if valor is None:
        return default
    try:
        if pd.isna(valor):
            return default
    except Exception:
        pass
    try:
        return str(int(round(float(valor))))
    except Exception:
        return str(valor)

def make_tooltip_html(row):
    tempo_medio_seg = row.get("tempo_medio_servicos_seg", np.nan)
    tempo_minimo_seg = row.get("tempo_minimo_servico_seg", np.nan)

    dist_media = row.get("distancia_media_servicos", np.nan)
    dist_minima = row.get("distancia_minima_servico", np.nan)
    dist_rede = row.get("dist_to_network", np.nan)

    tempo_medio_text = fmt_minutos_from_seg(tempo_medio_seg)
    tempo_minimo_text = fmt_minutos_from_seg(tempo_minimo_seg)

    dist_media_text = fmt_metros(dist_media)
    dist_min_text = fmt_metros(dist_minima)
    dist_rede_text = fmt_metros(dist_rede)

    osm_id_text = limpar_valor(row.get("osm_id", "N/A"))
    servico_min_id_text = limpar_valor(row.get("servico_min_id", "N/A"), max_len=80)
    servico_min_categoria_text = limpar_valor(row.get("servico_min_categoria", "N/A"), max_len=80)
    component_id_text = limpar_valor(row.get("component_id", "N/A"))
    nearest_node_text = limpar_valor(row.get("nearest_node", "N/A"), max_len=80)
    servicos_por_categoria_text = limpar_valor(row.get("servicos_por_categoria", "N/A"), max_len=200)

    status_text = "Sem serviço válido" if row.get("sem_servico_valido", False) else "Com serviço válido"

    return f"""
    <div style="font-size: 12px; max-width: 420px;">
    <b>OSM ID:</b> {osm_id_text}<br>
    <b>Status:</b> {status_text}<br><br>

    <b>--- ACESSIBILIDADE ---</b><br>
    <b>Distância mínima ao serviço:</b> {dist_min_text}<br>
    <b>Tempo mínimo ao serviço:</b> {tempo_minimo_text}<br>
    <b>Serviço mais próximo:</b> {servico_min_id_text}<br>
    <b>Categoria do serviço mais próximo:</b> {servico_min_categoria_text}<br><br>

    <b>--- MÉDIAS ---</b><br>
    <b>Distância média aos serviços:</b> {dist_media_text}<br>
    <b>Tempo médio aos serviços válidos:</b> {tempo_medio_text}<br><br>

    <b>--- DISPONIBILIDADE ---</b><br>
    <b>Número de serviços próximos:</b> {fmt_int(row.get('numero_servicos_proximos', np.nan), default='N/A')}<br>
    <b>Centro de Saúde:</b> {fmt_int(row.get('Centro Saude', np.nan), default='N/A')}<br>
    <b>Farmácias:</b> {fmt_int(row.get('Farmacias', np.nan), default='N/A')}<br>
    <b>Hospitais:</b> {fmt_int(row.get('Hospitais', np.nan), default='N/A')}<br>
    <b>Supermercados:</b> {fmt_int(row.get('Supermercados', np.nan), default='N/A')}<br>
    <b>Bancos:</b> {fmt_int(row.get('Bancos', np.nan), default='N/A')}<br>
    <b>CTT:</b> {fmt_int(row.get('CTT', np.nan), default='N/A')}<br>
    <b>Parques e jardins:</b> {fmt_int(row.get('Parques e jardins', np.nan), default='N/A')}<br><br>

    <b>--- DEBUG REDE ---</b><br>
    <b>Distância à rede:</b> {dist_rede_text}<br>
    <b>Component ID:</b> {component_id_text}<br>
    <b>Nearest node:</b> {nearest_node_text}<br><br>

    <b>Serviços por categoria:</b> {servicos_por_categoria_text}
    </div>
    """

def add_legend_html(map_obj, legend_id, title, colors, values):
    grades = []
    for c, v in zip(colors, values):
        grades.append(
            f'<i style="background:{c};width:14px;height:14px;display:inline-block;margin-right:6px;"></i>{v}<br>'
        )

    legend_html = f"""
    <div id="{legend_id}" style="
        position: fixed;
        bottom: 35px;
        left: 35px;
        z-index: 9999;
        background-color: rgba(20,20,20,0.92);
        border: 1px solid #999;
        border-radius: 6px;
        padding: 10px 12px;
        font-size: 12px;
        color: white;
        max-width: 260px;
        box-shadow: 0 0 10px rgba(0,0,0,0.5);
    ">
    <div style="font-weight: bold; margin-bottom: 6px;">{title}</div>
    {''.join(grades)}
    </div>
    """
    map_obj.get_root().html.add_child(folium.Element(legend_html))

def value_label(v, col):
    if "tempo" in col:
        return fmt_minutos_from_seg(v)
    if any(k in col for k in ["dist", "Dist", "distancia"]):
        return fmt_metros(v)
    return fmt_int(v, default="N/A")

# ============================================================
# 2. LER DADOS
# ============================================================
df = pd.read_csv(csv_resultados)

if "geometry_wkt" not in df.columns:
    raise ValueError("O CSV não tem a coluna geometry_wkt.")

df["geometry"] = df["geometry_wkt"].apply(wkt.loads)
resultados_unidos = gpd.GeoDataFrame(df, geometry="geometry", crs="EPSG:3763")

# garantir tipos numéricos
for col in VARIAVEIS_MAPA.values():
    if col in resultados_unidos.columns:
        resultados_unidos[col] = pd.to_numeric(resultados_unidos[col], errors="coerce")

for col in ["dist_to_network", "component_id", "numero_servicos_proximos"]:
    if col in resultados_unidos.columns:
        resultados_unidos[col] = pd.to_numeric(resultados_unidos[col], errors="coerce")

# marcar sem serviço válido
if "tempo_medio_servicos_seg" in resultados_unidos.columns:
    resultados_unidos["sem_servico_valido"] = (
        (resultados_unidos["numero_servicos_proximos"].fillna(0) == 0) |
        (resultados_unidos["tempo_medio_servicos_seg"].isna())
    )
else:
    resultados_unidos["sem_servico_valido"] = (
        resultados_unidos["numero_servicos_proximos"].fillna(0) == 0
    )

resultados_mapa = resultados_unidos.to_crs(epsg=4326)

serv = pd.read_csv(csv_servicos)
serv = serv.dropna(subset=["Longitude", "Latitude"]).copy()
servicos_mapa = gpd.GeoDataFrame(
    serv,
    geometry=gpd.points_from_xy(serv["Longitude"], serv["Latitude"]),
    crs="EPSG:4326"
)

# ============================================================
# 3. MAPA BASE ESCURO
# ============================================================
centroides_proj = resultados_unidos.geometry.centroid
centroides_wgs84 = gpd.GeoSeries(centroides_proj, crs="EPSG:3763").to_crs(epsg=4326)

centro_lat = centroides_wgs84.y.mean()
centro_lon = centroides_wgs84.x.mean()

m = folium.Map(
    location=[centro_lat, centro_lon],
    zoom_start=14,
    tiles="CartoDB dark_matter",
    control_scale=True
)

# ============================================================
# 4. CAMADAS POR VARIÁVEL
# ============================================================
cmap = plt.get_cmap("YlOrRd")
legend_ids = []

for nome_variavel, col_valor in VARIAVEIS_MAPA.items():
    if col_valor not in resultados_mapa.columns:
        continue

    valores_validos = resultados_mapa[col_valor].dropna()
    if len(valores_validos) == 0:
        continue

    vmin = float(valores_validos.min())
    vmax = float(valores_validos.max())

    if vmin == vmax:
        vmax = vmin + 1e-9

    norm = plt.Normalize(vmin=vmin, vmax=vmax)

    def get_color(row, col=col_valor):
        if row.get("sem_servico_valido", False):
            return "#808080"
        valor = row.get(col, np.nan)
        if pd.isna(valor):
            return "#808080"
        return to_hex(cmap(norm(valor)))

    fg = folium.FeatureGroup(name=nome_variavel, show=(nome_variavel == VAR_DEFAULT))

    for _, row in resultados_mapa.iterrows():
        cor = get_color(row)
        tooltip_text = make_tooltip_html(row)

        gj = folium.GeoJson(
            row.geometry.__geo_interface__,
            style_function=lambda feature, color=cor: {
                "fillColor": color,
                "color": "black",
                "weight": 1,
                "fillOpacity": 0.7
            },
            highlight_function=lambda feature: {
                "weight": 2,
                "color": "cyan",
                "fillOpacity": 0.9
            },
            tooltip=folium.Tooltip(tooltip_text, sticky=True)
        )
        gj.add_to(fg)

    fg.add_to(m)

    # legenda própria da variável
    steps = np.linspace(vmin, vmax, 6)
    colors = [to_hex(cmap(norm(v))) for v in steps]
    values = [value_label(v, col_valor) for v in steps]

    legend_id = f"legend_{col_valor.replace(' ', '_')}"
    legend_ids.append(legend_id)

    add_legend_html(
        m,
        legend_id=legend_id,
        title=nome_variavel,
        colors=colors,
        values=values
    )

# esconder todas as legendas exceto a default
for nome_variavel, col_valor in VARIAVEIS_MAPA.items():
    legend_id = f"legend_{col_valor.replace(' ', '_')}"
    display_style = "block" if nome_variavel == VAR_DEFAULT else "none"
    script = f"""
    <script>
    document.addEventListener("DOMContentLoaded", function() {{
        var el = document.getElementById("{legend_id}");
        if (el) {{
            el.style.display = "{display_style}";
        }}
    }});
    </script>
    """
    m.get_root().html.add_child(folium.Element(script))

# ============================================================
# 5. SERVIÇOS
# ============================================================
servicos_layer = folium.FeatureGroup(name="Serviços", show=False)

for _, row in servicos_mapa.iterrows():
    popup_name = limpar_valor(row["Name"], max_len=80) if "Name" in row.index else "Serviço"
    popup_cat = limpar_valor(row["Category"], max_len=50) if "Category" in row.index else ""

    popup_text = f"{popup_name}"
    if popup_cat:
        popup_text += f"<br>Categoria: {popup_cat}"

    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=4,
        color="cyan",
        fill=True,
        fillColor="cyan",
        fillOpacity=0.9,
        weight=1,
        popup=popup_text
    ).add_to(servicos_layer)

servicos_layer.add_to(m)

# ============================================================
# 6. CONTROLO
# ============================================================
folium.LayerControl(collapsed=False).add_to(m)

# ============================================================
# 7. JS PARA MOSTRAR A LEGENDA CERTA
# ============================================================
# nomes das layers como aparecem no controlo
js_map = {nome: col for nome, col in VARIAVEIS_MAPA.items()}

js_code = f"""
<script>
document.addEventListener("DOMContentLoaded", function() {{

    function hideAllLegends() {{
        {''.join([f'''
        var el_{i} = document.getElementById("legend_{col.replace(" ", "_")}");
        if (el_{i}) el_{i}.style.display = "none";
        ''' for i, col in enumerate(VARIAVEIS_MAPA.values())])}
    }}

    function showLegendByLayer(name) {{
        hideAllLegends();
        var mapping = {js_map};
        var col = mapping[name];
        if (!col) return;
        var el = document.getElementById("legend_" + col.replaceAll(" ", "_"));
        if (el) el.style.display = "block";
    }}

    // observar cliques no controlo de camadas
    document.querySelectorAll(".leaflet-control-layers-selector").forEach(function(input) {{
        input.addEventListener("change", function() {{
            var label = this.parentElement.textContent.trim();
            showLegendByLayer(label);
        }});
    }});
}});
</script>
"""
m.get_root().html.add_child(folium.Element(js_code))

m

In [ ]:
# 1. garantir que não há zeros
df[df["tempo_minimo_servico_seg"] == 0]

# 2. garantir que não há missing
df[df["tempo_minimo_servico_seg"].isna()]

# 3. ver distribuição
df["tempo_minimo_servico_seg"].describe()

## Files produced by this notebook

For each parish run, the workflow produces:

1. four base subarea CSV files;
2. four topology-corrected final subarea CSV files;
3. one merged final parish CSV;
4. validation summaries and optional diagnostic maps.

The subarea files are intermediate outputs. For the public repository, retain the final parish CSV
and any intermediate files strictly required to rerun the topology-correction stage. After all seven
parishes have been processed, merge the seven final parish CSV files in the separate consolidation
notebook.